In [ ]:
import torch
from torch import nn

import triton
import triton.language as tl

DEVICE = triton.runtime.driver.active.get_active_torch_device()

@triton.jit
def _attention_forward_inner_mask(acc, l_i, m_i, q, desc_k, desc_v,
                             offset_y, dtype: tl.constexpr, start_m, qk_scale,
                             block_m: tl.constexpr, hidden_dim: tl.constexpr, block_n: tl.constexpr, stage: tl.constexpr,
                             offs_m: tl.constexpr, offs_n: tl.constexpr, n_ctx: tl.constexpr, non_mask: tl.constexpr, warp_specialize: tl.constexpr):
    # print(f"Stage : {stage}")
    if stage == 1:
        lo, hi = 0, start_m*block_m
    elif stage == 2:
        lo, hi = start_m*block_m, (start_m+1)*block_m
    else:
        lo, hi = 0, n_ctx

    if non_mask:
        lo, hi = 0, (start_m+1)*block_m

    offsetk_y = offset_y + lo
    offsetv_y = offset_y + lo
    for start_n in tl.range(lo, hi, block_n, warp_specialize=warp_specialize):
        # print(f"start_n : {start_n} : offset of k [{offsetk_y},0]")
        k = desc_k.load([offsetk_y,0]).T
        qk = tl.dot(q, k)
        if stage == 2:
            mask = offs_m[:, None] >= (start_n + offs_n[None, :])
            # print(f"mask is used {mask}")
            qk = qk * qk_scale + tl.where(mask, 0, -1.0e6)
            m_ij = tl.maximum(m_i, tl.max(qk, 1))
            qk -= m_ij[:, None]
        else:
            # print("no mask used")
            m_ij = tl.maximum(m_i, tl.max(qk, 1) * qk_scale)
            qk = qk * qk_scale - m_ij[:, None]
        p = tl.math.exp2(qk)
        # -- compute correction factor
        alpha = tl.math.exp2(m_i - m_ij)
        l_ij = tl.sum(p, 1)

        acc = acc * alpha[:, None]

        # print(f"Offset of v [0, {offsetv_y}]")
        v = desc_v.load([offsetv_y, 0])
        p = p.to(dtype)
        acc = tl.dot(p, v, acc)
        l_i = l_i * alpha + l_ij
        m_i = m_ij
        offsetk_y += block_n
        offsetv_y += block_n
    return acc, l_i, m_i

@triton.jit
def _attention_forward(sm_scale, max_tensor, num_heads, n_ctx, desc_q, desc_k, desc_v, desc_o,
                       hidden_dim: tl.constexpr, block_m: tl.constexpr, block_n: tl.constexpr, mask_region: tl.constexpr,
                       warp_specialize: tl.constexpr):
    dtype = tl.float32
    assert block_n <= hidden_dim
    start_m = tl.program_id(0)
    off_h = tl.program_id(1)
    # print(f"start_m : {start_m}, off_h : {off_h}")
    y_dim = num_heads * n_ctx
    desc_q = tl.make_tensor_descriptor(desc_q, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                     block_shape=[block_m, hidden_dim])
    desc_v = tl.make_tensor_descriptor(desc_v, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                         block_shape=[block_n, hidden_dim])
    desc_k = tl.make_tensor_descriptor(desc_k, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                     block_shape=[block_n, hidden_dim])
    desc_o = tl.make_tensor_descriptor(desc_o, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                     block_shape=[block_m, hidden_dim])
    offset_y = off_h*n_ctx*hidden_dim
    # print(f"offset_y : {offset_y}")
    qo_offset_y = offset_y + start_m*block_m
    # print(f"qo_offset_y : {qo_offset_y}")
    offs_m = start_m*block_m + tl.arange(0, block_m)
    offs_n = tl.arange(0, block_n)
    # print(f"offs_m : {offs_m}")
    # print(f"offs_n : {offs_n}")
    # initialize pointer to m and l
    m_i = tl.zeros([block_m], dtype=tl.float32) - float("inf")
    l_i = tl.zeros([block_m], dtype=tl.float32) + 1.0
    acc = tl.zeros([block_m, hidden_dim], dtype=tl.float32)
    # load scales
    qk_scale = sm_scale
    qk_scale *= 1.44269504 #1/log(2)
    q = desc_q.load([qo_offset_y,0])
    # print(f"q load : {[qo_offset_y, 0]}")
    if mask_region:
        acc, l_i, m_i = _attention_forward_inner_mask(acc, l_i, m_i, q, desc_k, desc_v, offset_y, dtype, start_m, qk_scale, block_m, hidden_dim, block_n, 1, offs_m, offs_n, n_ctx, False, warp_specialize)
        acc, l_i, m_i = _attention_forward_inner_mask(acc, l_i, m_i, q, desc_k, desc_v, offset_y, dtype, start_m, qk_scale, block_m, hidden_dim, block_n, 2, offs_m, offs_n, n_ctx, False, warp_specialize)
    else:
        acc, l_i, m_i = _attention_forward_inner_mask(acc, l_i, m_i, q, desc_k, desc_v, offset_y, dtype, start_m, qk_scale, block_m, hidden_dim, block_n, 2, offs_m, offs_n, n_ctx, True, warp_specialize)
    m_i += tl.math.log2(l_i)
    acc = acc / l_i[:, None]
    m_ptrs = max_tensor + off_h * n_ctx + offs_m
    tl.store(m_ptrs, m_i)
    desc_o.store([qo_offset_y, 0], acc.to(dtype))

num_heads = 2
n_ctx = 64
block_m = 32
block_n = 16
sm_scale = 1.3
hidden_dim = 64*num_heads
embedding = nn.Embedding(200021, hidden_dim, device=DEVICE)

embedded_tensor = embedding(torch.randint(low=0, high=200021, size=(n_ctx,), device=DEVICE))

q = embedded_tensor.reshape(n_ctx, num_heads, -1).permute(1, 0, 2).contiguous()
k = embedded_tensor.reshape(n_ctx, num_heads, -1).permute(1, 0, 2).contiguous()
v = embedded_tensor.reshape(n_ctx, num_heads, -1).permute(1, 0, 2).contiguous()

print(f"q{q.shape} strides : {q.stride()}")
print(f"k{k.shape} strides : {k.stride()}")
print(f"v{v.shape} strides : {v.stride()}")


o = torch.empty_like(q)
M = torch.empty((q.shape[0], q.shape[1]), device=DEVICE, dtype=torch.float32)
grid = (n_ctx//block_m, num_heads, 1)
_attention_forward[grid](sm_scale, M, num_heads, n_ctx,
                          q, k, v, o,
                          hidden_dim, block_m, block_n, True, True)
print(o)
# do = torch.rand_like(input_q)
# BLOCK_M = 64
# BLOCK_N = 32
# pre_block = 128
# num_hiddens = input_q.shape[-1]
# n_ctx = input_q.shape[1]
# grid = (input_q.shape[1]//pre_block, num_heads, 1)
# print(f"Grid : {grid}, q: {input_q.shape}, k: {input_k.shape}, v: {input_v.shape} ")
# delta = torch.empty((input_q.shape[0], input_q.shape[1]), device=input_q.device, dtype=torch.float32)
# # Preprocess
# _attention_bwd_pre_process[grid](o, do, delta, n_ctx, pre_block, num_heads, num_hiddens)
# print(delta)


tensor([[[-5.1848e-01, -1.7938e+00, -1.1350e+00,  ..., -3.6118e-02,
          -1.0831e+00,  1.5447e+00],
         [ 7.1633e-01,  1.3346e-01,  5.4062e-01,  ...,  4.6964e-01,
          -2.0940e-02,  2.2066e-01],
         [ 1.0506e+00, -1.8215e-01, -6.7207e-01,  ..., -7.4940e-01,
           2.2424e-02, -5.9976e-01],
         ...,
         [-2.5905e+00,  2.0470e+00,  4.1108e-01,  ...,  7.4265e-01,
           1.6003e+00, -1.2177e+00],
         [-1.3377e+00, -1.2713e-01, -3.9221e-01,  ...,  6.3038e-02,
          -1.1148e+00,  1.6521e+00],
         [-1.2372e+00,  1.3918e+00, -1.2659e+00,  ...,  7.9130e-02,
          -3.2453e-01,  2.6393e-01]],

        [[-1.1712e+00, -6.8080e-01, -3.7687e-01,  ...,  2.6303e-01,
           3.5718e-01, -4.2570e-01],
         [ 7.4012e-01,  1.7430e+00, -4.9001e-01,  ..., -1.1297e+00,
          -1.6528e+00,  7.2890e-01],
         [ 5.6520e-01, -1.0189e+00,  9.7232e-01,  ..., -7.6217e-01,
           9.9109e-01,  9.1670e-01],
         ...,
         [ 8.3901e-02, -8